# __Méthodologie en Génétique Humaine__

## **TP: Étude d'association pan-génomique dans le diabète de type 1**
### Activité préparatoire
*Claire Vandiedonck*



<u>Déroulé de l'ensemble du TP :</u> <i>cette activité préparatoire porte sur les parties 0 à 2 (en bleu)</i>

<blockquote><ul><span style="color:blue">
0. Synopsis<br>
1. Première partie: Evaluation de la faisabilité de l'étude<br><ul>
    1.A. Identification des régions déjà connues pour leur association au DT1 de manière significative<br>
    1.B. Calcul de puissance pour retrouver les mêmes régions ou d’autres variants avec effets similaires dans une nouvelle cohorte<br></ul>
2. Deuxième partie: Formatage de fichiers de génétique et découverte du logiciel PLINK
</span><br>
3. Troisième partie: Statistiques descriptives et QC<br>
4. Quatrième partie: Analyse d'association génétique cas-contrôles<br>
5. Cinquième partie: Analyse d'association familiale</blockquote>



## Avant d'aller plus loin
---
<div class="alert alert-block alert-danger"><b>Attention:</b> 
Ne travaillez pas directement sur ce notebook pour ne pas le perdre. Dupliquez-le et renommez-le par exemple en ajoutant vos initiales et travaillez sur cette nouvelle copie. Pour ce faire, dans le panneau de gauche, faites un clic droit sur le fichier et sélectionnez "Duplicate". Puis, toujours dans la colonne de gauche, faites un clic droit sur cette copie et sélectionnez "rename" pour changer le nom. Ouvrez ensuite cette nouvelle version en double cliquant dessus. Vous êtes prêt(e) à démarrer! <br>
<br>
<b>N'oubliez pas de sauvegarder régulièrement votre notebook</b>: <kbd>Ctrl</kbd> + <kbd>S</kbd>. ou en cliquant sur l'icone 💾 en haut à gauche de votre notebook ou dans le Menu du JupyterLab "File puis "Save Notebook"!
</div>

<div class="alert alert-block alert-info"> 
   
<b>Rappel :</b> Vous pouvez prendre des notes directement dans ce notebook, en ajoutant une cellule Markdown en cliquant sur l'icône <kbd>➕</kbd> dans la barre des menus, et en choisissant son format dans le menu déroulant. 
Les astuces pour utiliser les cellules des notebooks sont rappelées tout en bas dans un cadre bleu ⏬.
</div>

Dans ce TP, vous lancerez chaque cellule l'une après l'autre dans l'ordre. Si vous revenez en arrière et rééxécuter une cellule, veillez à relancer les suivantes. Les cellules sont numérotées dans leur ordre d'éxécution.

__*=> A propos de ce jupyter notebook*__

Pour ce TP, nous aurons besoin d'éxécuter du code **bash** et **R**. Nous allons utiliser un notebook avec un noyau **python** et indiquer que les cellules doivent être éxécutées en bash (natif) en ajoutant `%%bash` au début de chaque cellule, et en ajoutant `%%R` au début de chaque cellule `R` après avoir installé et chargé `Rpy2` au début du notebook. Cela restera très lisible.

Avant de commencer, nous devons donc charger le module `Rpy2` avec la commande suivante.

In [ ]:
#cell 1
%load_ext rpy2.ipython

Vous pouvez éxécuter cette commande python pour connaître votre répertoire de travail puis lister ce qu'il contient.

In [ ]:
# cell 2
import os 
os.getcwd()

<div class="alert alert-block alert-warning"><b>Le résultat devrait ressembler à :</b>"/srv/home/mylogin/meg_m1_ghm_gwas" avec votre login. Si ce n'est pas le cas, changez de répertoire avec la commande "os.chdir('path')" ou applez votre enseignant à l'aide!</div>

Pour lister ce qu'il contient, vous pouvez tenter avec les 3 langages:
- en python : la commande affichera aussi les fichiers cachés commençant par un `.`.

In [ ]:
# cell 3
os.listdir()

- en bash:

In [ ]:
%%bash
# cell 4
ls

- en R:

In [ ]:
%%R
# cell 5
list.files()


## **0 - Synopsis**
---

### Objectifs du TP

Dans ce TP, vous allez réaliser une étude d’association génétique dans le diabète de type 1 (DT1) dans une cohorte française au moyen d’outils informatiques et statistiques (Marees  AT et al., 2018).
Les sujets d'étude proviennent d'une série de patients atteints de diabète de type 1 et de contrôles sains. Tous les échantillons ont été génotypés au moyen de la puce Illumina immunochip (Trynka et al., 2011, Parkes M. et al., 2013). Au cours de ce TP, vous serez amenés à:
- évaluer la la puissance de l'étude
- comprendre le format des fichiers de génotypage pour les logiciels d'analyse génétique
- effectuer des contrôles de qualité
- nettoyer/filtrer les données
- réaliser l'étude d'association pangénomique de type cas/contrôles et de type familial
- interpréter vos résultats en comparaison avec les articles publiés par le Type 1 Diabetes Genetics Consortium (Onengut-Gumuscu  S, et al., 2015, Robertson CC et al., 2021).


⚠️ <mark>Vous aurez 2 notebooks: un pour l'activité préparatoire (celui-ci) et un pendant la séance de TP</mark>

### Références
1. Trynka G, et al. Dense genotyping identifies and localizes multiple common and rare variant association signals in celiac disease. Nature Genetics 43, 1193–1201 (2011) doi.org/10.1038/ng.998<br>
2. Parkes M, et al. Genetic insight into common pathways and complex relationships among immune-mediated diseases. Nature Genetics 14, 661-673 (2013) doi.org/10.1038/nrg3502<br>
3. Marees AT et al. A tutorial on conducting genome‐wide association studies: Quality control and statistical analysis. Int J Methods Psychiatr Res. 27:e1608. (2018) doi.org/10.1002/mpr.1608<br>
4. Onengut-Gumuscu S et al. Fine mapping of type 1 diabetes susceptibility loci and evidence for colocalization of causal variants with lymphoid gene enhancers. Nature Genetics 47, 381–386 (2015) doi.org/10.1038/ng.3245<br>
5. Robertson CC et al. (2021). Fine-mapping, trans-ancestral and genomic analyses identify causal variants, cells, genes and drug targets for type 1 diabetes. Nature Genetics 53, 962–971 (2021). doi.org/10.1038/s41588-021-00880-5




## **I - Première partie: évaluation de la faisabilité de l'étude**
---

Avant d'entamer toute étude de génétique, il est nécessaire d'évaluer l'état des connaissances et de vérifier si l'étude prévue aura la puissance nécessaire pour identifier des associations génétiques.


### **1.A. Quelles sont les régions génétiques déjà associées au DT1 ?**

Pour répondre à cette question, vous allez étudier le résultats publiés par par Onengut-Gumuscu et al. Nature Genetics (2015), présentant les derniers résultats de GWAS publiés par le consortium international de l'étude du diabète de type 1 (T1DGC for Type 1 Diabetes genetics Consortium). L'article au format .pdf est présent sur moodle et dans votre environnement jupyterlab.

<span style="color:blue"><b>Q1.1.</b><i> Combien de <b>régions génomiques</b> avaient été impliquées dans le diabète de type I (DT1) avant le démarrage de l'étude ?</i></span>

*votre réponse...*

<span style="color:blue"><b>Q1.2.</b><i> Dans quels buts a été conçue l'<b>Immunochip</b> ? Quelle a été la stratégie de conception de l'Immunochip ?</i></span>

*votre réponse...*

<span style="color:blue"><b>Q1.3.</b><i> Quelle est la <b>structure des échantillons</b>? Combien de cas et de contrôles ont été étudiés après QC (Quality Controls sur les SNPs et sur les sujets) ? Quelle est leur origine géographique?</i></span>

*votre réponse...*

<span style="color:blue"><b>Q1.4.</b><i> Combien de <b>SNPs</b> ont passé les seuils de QC ? </i></span>

*votre réponse...*

<span style="color:blue"><b>Q1.5.</b><i> Avec R, calculer la valeur de <b>seuil de Bonferroni pour le nombre de SNPs ayant passé les QCs</b>. S'agit-il du seuil utilisé dans l'artcile? </i></span>

In [ ]:
%%R
#cell 6 et votre code R

*votre réponse...*


<span style="color:blue"><b>Q1.6.</b><i> En dehors du CMH qui n'est pas étudié dans cet article, <b>combien de régions ont été trouvées associées</b> à leur seuil? En bas de la colonne 2 de la 1ère page de l'article, combien de régions les auteurs répliquent-ils et combien de nouvelles régions sont associées au seuil 5e-8 ?</i></span>

*votre réponse...*

<span style="color:blue"><b>Q1.7.</b><i> Quelle <b>version de la séquence de référence</b> du génome est utilisée dans cet article?</i></span>

*votre réponse...*


<div class="alert alert-block alert-warning"><b>Pour information :</b><br>
<b>Jusque juin 2021</b>, l'ensemble des études pangénomiques sur le DT1 réalisées essentiellement par le T1DGC (Type 1 Diabetes Genetics Consortium) avait permis d'identifier 52 régions associées de manière significative au seuil pangénomique 5x10-8. Cinq autres régions avaient été associées de manière suggestives au seuil 5x10-5  : 2q11.2, 6q23.3, 11q13.1, 14q24.1 et 17q21.31, les 4 premières étant associées de manière significative à une autre maladie autoimmune ou inflammatoire.<br>
<b>A ce jour, 78 régions significatives sont connues </b>. En effet, 36 nouvelles régions ont été indetifiées comme présentant une association significative dans la dernière étude d'association pangéomique publiée en juin dernier (Robertson CC, Nature Genetics 2021). Cette étude doublait la taille de l'échantillon par rapport aux précédentes  avec plus de 61 000 sujets de plusieurs ascendances (Européenne, AFricaine, Finnoise, Est-Asiatiques et autres populations mélangées). L'étude portait toujours sur le génotypage au moyen de l'immunochip et a inclus également l'imputation de 137k à 322k SNPs selon les populations. La publication est disponible sur moodle.
</div>

### **I.B. Quelle est la puissance de détecter les mêmes associations ou de nouvelles associations avec des effets comparables ?**

Dans cette partie, vous allez évaluer la capacité de répliquer les effets déjà publiés dans une cohorte française génotypée au moyen de l'immunochip. Vous réaliserez le GWAS de cette cohorte dans les prochaines séances de TP.

Pour rappel, dans un test d'association génétique:

- `l'hypothèse nulle (H0)` correspond à l'absence d'association, c'est à dire qu'il n'y a pas d'effet: les fréquences alléliques sont identiques entre cas et contrôles.

- `l'hypothèse alternative (H1)` est celle de l'absence d'égalité des fréquences alléliques: il existe un effet, c'est à dire une association allélique.

En statistiques, on définit la `puissance` comme la probabilité de detecter un effet s'il existe. Il s'agit de la probabilité de rejeter H0 sachant que H0 n'est pas vraie. Il faut connaitre une hypothèse H1 spécifique, c'est-à-dire connaitre l'amplitude de l'effet en question. Dans une étude d'association génétique, l'OR correspond à l'effet. 

Dans le cas d'un test de comparaison de fréquences, il faut également connaître la fréquence de chaque groupe, cas et contrôles sous H1. Connaissant la fréquence de contrôles et l'OR, vous pouvez déduire la fréquence des cas en admettant des effectifs de taille similaire entre cas et contrôles (voir formule plus bas).

Si vous ne génotypez pas directement le SNP reporté comme étant associé, mais un SNP en déséquilibre de liaison (DL) avec lui, certains outils de calcul de puissance vous demandent de renseigner ce DL. Dans ce TP, nous espérons que le variant associé est bien inclus dans l'immunochip. Nous supposerons donc que nous testons directement le variant associé pour les calculs de puissance.

<div class="alert alert-block alert-warning"><b>En pratique</b><br>Vous allez calculer la puissance pour une taille d'échantillon donnée que vous ferez varier de 50 à 5 000 (pour chaque groupe, cas et contrôles) pour les deux meilleures régions associées au DT1 à ce jour après le CMH:<ul>
    <li>la région du gène de l'Insuline (<i>INS</i>) en 11p15.5:  chr11:2092701-2260001</li>
    <li>le gène <i>PTPN22</i> en 1p13.2: chr1:113288123-114009223</li></ul>
</div>


- **collecte des informations des 3 meilleures régions associées avant calcul de puissance:**

Pour calculer la puissance, vous devez collecter des informations concernant les régions déjà associées.

A titre de référence et pour comparaison, nous vous fournissons les valeurs pour la région du CMH (en 6p21.3 : chr6:32350867-32714887) et les résultats du calcul de puissance dans les tableaux 1 et 2 ci-dessous que vous complèterez.

<center><b>Tableau 1.</b> Valeurs de référence pour les top SNPs des 3 meilleurs régions associées au DT1</center>

| **région** |   **SNP** | **allèles** | **p-value** | **OR pour mineur** |                    **OR pour majeur** | **MAF** | **freq majeur** |                            **freq allèle mineur patients** | **freq allèle majeur patients** |
|-----------:|----------:|------------:|------------:|-------------------:|--------------------------------------:|--------:|----------------:|-----------------------------------------------------------:|--------------------------------:|
|        CMH | rs6916742 |         C>T |      <e-100 |               0.24 | 4.17                          =1/0.24 |    0.39 |            0.61 | 0.13                          =0.39*0.24(1-0.39+0.39*0.24) |                            0.87 |
|        INS |           |             |             |                    |                                       |         |                 |                                                            |                                 |
|     PTPN22 |           |             |             |                    |                                       |         |                 |                                                            |                                 |


<span style="color:blue"><b>Q1.8.</b><i> Pour les deux régions, complétez les colonnes du Tableau 1 à partir des valeurs de l'article Onengut-Gumuscu et al. 2015 :</span><ul>
    <span style="color:blue"><li>le top SNP (celui qui a la p-value la plus faible) et ses allèles Majeur>Mineur</li>
    <li>la p-value de l'association au DT1</li>
    <li>l'OR de risque correspondant</li>
    <li>la fréquence de l'allèle de risque chez les contrôles de l'étude</i></li></ul></span>

*Entrez ici d'éventuelles explications pour compléter le tableau...*


- **Calcul de puissance pour les 3 meilleures régions associées:**

A présent vous pouvez calculer la puissance de détecter ces mêmes effets ou des effets d'amplitude similaire, connaissant la fréquence de l'allèle de risque associé et l'OR.

Il existe plusieurs outils permettant de calculer la puissance dans le cas d'un test d'association génétique, tels que:
<br>- Genetic Power Calculator : http://zzz.bwh.harvard.edu/gpc/
<br>- Quanto : http://biostats.usc.edu/Quanto.html
<br>- GAS Power Calculator: https://csg.sph.umich.edu/abecasis/gas_power_calculator/index.html

Leur utilisation n'est pas si simple car elle ne prend pas en compte les 3 seuls paramètres: OR, fréquence et taille pour déterminer la puissance. Ils prennent également en paramètre d'entrée la prévalence de la maladie (afin de déduire le nombre de contrôles en réalité atteints). Ils peuvent considérer si le SNP testé est le variant causal ou s'il est en DL avec le causal. Il faut alors renseigner soir le DL soit les fréquences génotypiques ou alléliques au SNP testé en plus du causal. Ces outils peuvent aussi prendre en compte différents modèles (additif, dominant, etc...) voire des effets de l'environnement et des interactions. De plus, ils ne prennent pas directement en compte l'OR mais d'autres valeurs d'effet comme le risque relatif par génotype.

Dans ce TP, nous avons décidé d'utiliser un **pw, un paquet de R dédié au calcul de puissance pour les tests statistiques classiques** (http://cran.r-project.org/web/packages/pwr/index.html). Ce paquet présente un double avantage: (i) il est générique donc utilisable  en dehors d'un contexte génétique; (ii) il permet de générer une courbe de la puissance en fonction de la taille d'échantillon en une commande.

Chargeons le paquet pwr avec la commande suivante:

In [ ]:
%%R
# cell 7
library(pwr)
sessionInfo()



Nous l'utiliserons dans le contexte d'un **test statistique du Chi2 à 1 degré de liberté** que nous pouvons utiliser pour un test allélique. La fonction est `pwr.chisq.test()`.
Les arguments que nous devons renseigner sont:

- `N`: la taille totale des échantillons 
- `sig.level` : le degré de signification (vous utiliserez le seuil de p-value corrigé pour un GWAS, c'est-à-dire 5x10-8)
- `w`: l'effect size selon la définition de Cohen.

L'`effect size` peut se mesure de différentes façons dans un test statistique. En génétique, on choisit plutôt l'OR que la différence entre les fréquences alléliques théoriques chez les cas et contrôles. Plus classiquement, cette mesure de l'amplitude de l'effet peut être mesurée par la valeur `d` de Cohen. C'est celle qui est implémentée dans ce paquet. Il n'est pas nécessaire de rentrer dans le détail de cette mesure mais vous pouvez trouver au lien suivant un outil en ligne pour obtenir les conversions d'un OR en valuer d de Cohen ou tout autre mesure de la taille de l'effet: https://www.escal.site/

Nous calculerons l'effect size `d` directement dans R au moyen de la formule le reliant à l'OR:</p> 
						
> d <- sqrt(3)*ln(OR)/pi

Voici ci-dessous les commandes pour calculer la puissance pour la région du CMH à partir d'une taille d'échantillon de 100 et pour générer la courbe puissance ~ taille de l'échantillon:

In [ ]:
%%R
# cell 8
d_cmh <- sqrt(3)*log(4.17)/pi
print(d_cmh)
print(pwr.chisq.test(w = d_cmh, df = 1, N = 100, sig.level = 5e-8))

plot(pwr.chisq.test(w = d_cmh, df = 1, power = 0.80, sig.level = 5e-8), main="CMH")

<span style="color:blue"><b>Q1.9.</b><i> Pour le CMH et les deux régions des gènes de l'INS et PTPN22, complétez les colonnes du Tableau 2 à partir des valeurs de l'article Onengut-Gumuscu et al. 2015 et tracez la courbe puissance ~ taille échantillon.</i></span>

<center><b>Tableau 2.</b> Puissance pour les différents effectifs</center>

| **région** | **50** | **100** | **200** | **300** | **500** |
|-----------:|-------:|--------:|--------:|--------:|--------:|
|        CMH |    55% |     99% |    100% |    100% |    100% |
|        INS |        |         |         |         |         |
|     PTPN22 |        |         |         |         |         |

In [ ]:
%%R
#cell 9 pour les calculs

In [ ]:
%%R
# cell 10, votre code R pour les figures

<span style="color:blue"><b>Q1.10.</b><i> Commentez les résultats.</i></span>

<div class="alert alert-block alert-warning"><b>En pratique</b><br>
⚠️  On calcule la puissance avant tout projet afin de connaitre la taille des échantillons recquise pour mettre en évidence un effet minimum. Cependant, on recalcule la puissance à la fin de l'étude en fonction des fréquences alléliques effectivement estimées dans notre échantillon afin de s'assurer, en cas d'absence d'association, qu'on avait bien la puissance nécessaire. Pensez-y pour votre compte-rendu!</div>

<div class="alert alert-block alert-success"><b>=> Bravo!</b><br>

Vous êtes désormais prêts pour mener le GWAS "cas-contrôles". mais nous devons encore apprendre à utiliser le logiciel PLINK pour mener cette étude.</div>

---
---

## **II. Deuxième partie: format de fichiers de génétique et découverte du logiciel PLINK**
---
    
### **II.A. Le logiciel PLINK**

Pour réaliser l'étude d'association, vous devrez préparer les fichiers sous un format utilisable par les logiciels de génétique. Le principal logiciel que vous utiliserez est **PLINK**: https://www.cog-genomics.org/plink/1.9/.

En plus de permettre des analyses d'association, ce logiciel permet de manier les fichiers de génétique, de faire des sélections et filtrages, et de réaliser certains contrôles de qualité. Il est mondialement reconnu comme un des meilleurs outils pour réaliser des GWASs.
Ce logiciel libre existe en versions windows, mac et unix.
Le tutoriel (https://www.cog-genomics.org/plink/2.0/tutorials/tutorial_setup) est très détaillé et donne de nombreux exemples.

Ce logiciel a été développé par Shaun Purcell. Il avait alors une double affiliation au Center for Human Genetic Research (CHGR), Massachusetts General Hospital (MGH) et au Broad Institute of Harvard & MIT.  Il est désormais chef de son labo (https://zzz.nyspi.org/). La version originale présente sur son site web était la version v1.07 (10-Oct-2009).

Il existe à présent une nouvelle version beta **PLINK 1.9** dont l'objectif est principalement d'améliorer la vitesse et la fonctionnalité pour de très gros jeux de données. Elle a été principalement développée par  par Christopher Chang at Human Longevity, Inc., par Carson Chow and Shashaank Vattikuti at the NIH-NIDDK's Laboratory of Biological Modeling, Laurent Tellier at the BGI Cognitive Genomics Lab, and James Lee at the University of Minnesota, with additional funding from the Purcell Lab at Mount Sinai School of Medicine du BGI Cognitive Genomics Lab! 

> Dans ce TP, vous utiliserez la version **PLINK 1.9** (https://www.cog-genomics.org/plink/1.9/) désormais stable (beta 6.24, 6 Jun 2021) qui a été installée sur adénine. Pour note, il existe une version PLINK2.0 en cours de développement.

<div class="alert alert-block alert-warning"><b>En pratique</b><br>PLINK s'utilise en lignes de commandes « relativement simples ». 
Toutes les informations pour spécifier les fichiers d'entrée (« input »), les fichiers de sortie (« output ») et les paramètres d'utilisation sont détaillés dans le menu à gauche de la page d’accueil de PLINK. </div>

### **II.B. Lancement de PLINK**

Pour faire tourner PLINK sous Unix ou dans ce notebook, vous appelez d'abord le programme en tapant : `plink` dans une cellule **bash**. 

<span style="color:red"><i>N'ayez pas peur de la boite rouge qui apparait. Cet affichage disparaitra quand les commandes seront complètes.</i></span>

In [ ]:
%%bash
## cell 11
plink

Le programme démarre et s'arrête avec une erreur car vous n'avez pas encore donné de fichiers d'entrée (*inputs*).

>- Si toutefois vous vouliez utiliser PLINK en dehors d'adenine, voici des conseils d'utilisation **sous Windows**:<br>
>Vous devez utiliser le DOS en utilisant l'invite de commande (cmd.exe) ou, sur les dernières versions de windows, le "windows power shell" à partir du menu "fichier" de votre répertoire de travail.  Vos fichiers "inputs" doivent être placés dans le même répertoire (=dossier) que l'éxécutable de PLINK plink.exe.
>Trois options s'offrent à vous, de la plus compliquée à la plus simple!:
>
>     **(1)** Dans le menu « Démarrer » de Windows, rechercher « cmd » puis indiquez le chemin du répertoire où est situé plink.exe (pour changer de répertoire, la commande est “cd”; pour passer au répertoire supérieur, taper “cd ..” ; vous pouvez utiliser la flèche de tabulation à gauche du clavier pour autocompléter les noms de dossiers)
> 
>     **(2)** Alternativement, trouvez le fichier cmd.exe (en général dans Windows\system 32; vous pouvez le trouver aussi dans les accessoires). Faire une copie que vous placez dans le dossier contenant plink.exe. Si vous double-cliquez alors sur cmd.exe dans ce répertoire, vous pouvez utiliser PLINK.
>
>     **(3)** Encore plus simplement, vous pouvez aussi créer en local un répertoire dans lequel vous mettrez tous vos documents du TP (fichiers d'input, d'output) ainsi que vos programmes. Copiez et collez l'executable plink.exe dans ce répertoire. Copiez et collez également l'exécutable de l'invite de commande cmd.exe dans ce répertoire. Si vous double-cliquez alors sur cmd.exe dans ce répertoire, vous pouvez utiliser PLINK.
>
>      Alternativement, si vous travaillez depuis windows power shell, il faut spécifier que plink est dans le répertoire de travail en tapant: .\plink
      

### **II.C. Fichiers d'entrée**

Commençons par nous familiariser avec l'utilisation avec des fichiers « test ».

PLINK a besoin de <u>deux types de fichiers d'entrée.</u>

- `.ped` pour le fichier de structure de pedigree
- `.map` pour le fichier des marqueurs génotypés

Dans le répertoire `input` de votre environnement jupyterlab, vous avez des fichiers d'entrée d'exemple avec le préfixe `test.`. Les fichiers sont aussi disponibles sur moodle.

##### **1. Exploration des fichiers d'entrée en dehors de PLINK**

<span style="color:red">Ouvrez-les en double cliquant dessus dans le jupyterlab </span> (ou avec un éditeur de texte comme notepad++ sous windows ou gedit sous unix si vous ne travaillez pas sur adénine).
<b>Sans lancer PLINK</b>, cherchez des explications sur leur structure dans la documentation de PLINK à ce lien en bas du menu de gauche"https://www.cog-genomics.org/plink/1.9/formats). Ne vous arrêtez pas en haut de la page! Allez sur les paties dédiées aux ficg=hiers .bed et .map.



<span style="color:blue"><b>Q2.1</b><i>  Concernant le fichier de structure de <b>pedigree</b>:</i></span>
    
<span style="color:blue"><i>- Combien de sujets ont été génotypés ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Sont-ils apparentés ? Ont-ils les mêmes parents ? Quelles colonnes vous donnent ces informations ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Quel est l'identifiant du 5ème sujet ? Est-il possible de le distinguer des autres sujets en considérant un identifiant unique ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Quel est le sexe des sujets ? Quelle colonne vous donne cette information ? Est-ce cohérent avec les autres colonnes ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Connaît-on leur statut vis-à-vis de la maladie ? Combien y a-t-il de cas et de contrôles ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Combien de marqueurs ont été génotypés pour ces sujets ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Combien de colonnes contiendraient un fichier de pedigree avec n marqueurs génotypés ?</i></span>

*votre réponse*

<span style="color:blue"><b>Q1.2.</b><i> Concernant le fichier de <b>marqueurs</b> :</i></span>
    
<span style="color:blue"><i>- Combien y-a-t-il de marqueurs dans ce fichier ? Est-ce en accord avec le fichier de pedigree ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Sur quels chromosomes sont-ils situés ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Connaissez-vous leur position physique ? A partir de quelle origine ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Connaissez-vous leur position génétique ?</i></span>

*votre réponse*

##### **2.  Chargement des fichiers dans PLINK**

Nous allons à présent charger ces fichiers sans les modifier dans PLINK.<br> Il existe deux commandes possibles.

1. ```plink --file input/test```

ou

2. ```plink --ped input/test.ped --map input/test.map```

Dans la 1ère, nous utilisons l'argument `--file` qui nous permet de lire tous les fichiers inputs commençant par le préfixe `test` dans le répertoire `input`.
Dans la 2nde commande, nous utilisons un argument pour chaque type de fichier input. Cette seconce commande est plus longue à taper mais permet de charger des inputs qui n'auraient pas le même préfixe.

Exécutons ces 2 commandes l'une après l'autre. Elles retournent bien le même résultat. Une erreur apparaît dans les 2 cas. Elle était attendue. Nous reviendrons un peu plus bas sur cette erreur (partie II.E).

In [ ]:
%%bash
# cell 12
#la premiere commande
plink --file input/test

Notez que vous avez obtenu un fichier texte de sortie `plink.log` que vous pouvez ouvrir en double cliquant dessus. Il contient tout ce qui s'est affiché à la suite de la commande jusqu'à la ligne d'erreur.

In [ ]:
%%bash
# cell 13
#la deuxième commande
plink --ped input/test.ped --map input/test.map

Le fichier texte `plink.log` a été mis à jour et contient l'affichage des options/argument de cette 2ème commande. Voyons immédiatement comment  spécifier un nom aux fichiers de sortie pour ne pas les écraser à chaque utilisation de PLINK.

### **II.D. Spécification des noms de fichier de sortie**

A chaque commande PLINK, si vous spécifiez des arguments, des fichiers de sortie (*outputs*) sont automatiquement générés avec différentes extensions. En particulier, vous obtenez un fichier `.log` qui a sauvegardé tout ce qui s'était affiché dans votre terminal ou dans le notebook.

Vous pouvez spécifier le préfixe de ces outputs avec l'argument `--out`.

Dans PLINK, vous pouvez entrer plusieurs arguments à la suite sur la même ligne de commande. Si vous avez utilisé l'argument `-- out`, tous les autres outputs générés contiendront le même préfixe suivis de leur extension propre. On peut spécifier le répertoire dans lequel ils seront sauvergardés en indiquant le chemin.

On réessaye avec les 2 commandes précédentes en spécifiant un préfixe aux fichiers d'output que nous sauvegarderons dans le répertoire `output`

In [ ]:
%%bash
# cell 14
#la premiere commande avec la spécification de la sortie
plink --file input/test --out output/first

In [ ]:
%%bash
# cell 15
#la deuxième commande
plink --ped input/test.ped --map input/test.map --out output/second

Cette fois-ci, aucun fichier texte `plink.log` n'a été créé. Après la 1ère commande, un fichier `first.out` a été créé, après la seconde un fichier `second.out` a été créé. Ils sont dans le répertoire `output`. Vous pouvez les ouvrir. Notez qu'il contiennent désormais 2 lignes après « Options in effect ».

Un autre fichier a été créé après chaque commande avec l'extension `.fam`. Vous pouvez noter qu'ils contiennent le début du fichier `test.ped` présent dans `input`. Ils sont incomplets car la commande a été interrompue par l'erreur.

### **II.E. Correspondance des fichiers d'input**

<span style="color:blue"><b>Q2.3.</b><i> Quelle erreur avez-vous rencontrée avec la 1ère et la 2ème commande ?</i></span>

*Votre réponse...*

On apprend désormais que le fichier de pedigree ne contenait les génotypes que de marqueurs autosomiques. On a nettoyé le fichier afin de ne garder que les marqeurs autosomiques. Il a été sauvegardé sous un autre nom (`autosomic.map` déjà dans le répertoire `input`).

Chargez-le à présent dans PLINK avec le fichier de pedigree. On n'oublie-pas de changer le préfixe des sorties pour ne pas écraser le précédent fichier .log.

In [ ]:
%%bash
# cell 16, votre commande

<span style="color:blue"><b>Q2.4.</b><i> Les informations obtenues sont-elles désormais cohérentes avec ce que vous savez?</i></span>

*votre réponse*


<div class="alert alert-block alert-warning"><b>En pratique: </b><br> Cette structure de pedigree et de liste de marqueurs est utilisée par la plupart des logiciels de génétique. Il existe cependant quelques variantes, en particulier pour l'encodage des phénotypes parfois dans un troisième fichier, ou pour le fichier de marqueurs qui ne contient pas toujours la position.

<b>L'ordre des colonnes des génotypes dans le fichier de pedigree doit toujours correspondre à l'ordre des marqueurs. Il est donc fondamental de bien formater ses fichiers !</b>
</div>

### **II.F. Fichiers binaires**

PLINK peut travailler avec des formats de fichier de grande taille : nombreux sujets et nombreux génotypes.
<br>
- Dans ce cas, il existe une **version binaire des fichiers** moins gourmande en espace disque et en temps d'analyse. Consultez la section « Binary PED files » (https://www.cog-genomics.org/plink/1.9/formats#bed) et effectuez la conversion des fichiers test en format binaire en sauvegardant les outputs avec le préfixe `mytest_bdata` en utilisant l'argument `--make-bed`.


In [ ]:
%%bash
#cell 17, votre commande

<span style="color:blue"><b>Q2.5.</b><i>Concernant ce <b>format binaire</b> :</i></span>

<span style="color:blue"><i>- Quel fichier est converti en format réellement binaire ? Quelle est l'extension de ce fichier ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Quels autres fichiers ont été générés ? A quoi correspondent-ils ?</i></span>

*votre réponse*

<span style="color:blue"><i>- Quelle information n'était pas dans le fichier .map et est désormais présente dans le fichier de marqueurs ?</i></span>

*votre réponse*

**- Pour lire avec PLINK les fichiers binaires** ainsi générés, on utilise l'argument `--bfile` comme dans la commande suivante (qui s'interrompt car aucune manipulation ni output spécifique en dehors du .log n'a été demandé):

In [ ]:
%%bash
# cell 18
plink --bfile output/mytest_bdata --out output/load_mytest_bdata

La commande ci-dessus ne demandant pas d'output à partir des fichiers d'input binaire, aucune fichier de log n'est créé. Nous verrons dans la séance de TP que l'on peut demander une sortie par exemple da la liste de SNPs étudiés afin d'avoir malgré tout un log et de savoir notamment combien de sujets et de marqueurs ont été lus.

- Inversement, il existe un moyen de **convertir des données binaires en format non binaire** (cf. aide la section https://www.cog-genomics.org/plink/1.9/data) avec l'argument `--recode`

In [ ]:
%%bash
# cell 19
plink --bfile output/mytest_bdata --recode --out output/mytest_data

<span style="color:red"><i>Ne faites pas attention aux chiffres affichés dans la ligne commençant par "Calculating allele frequencies" si vous lisez le log directement sur le notebook. Cette même ligne a une apparence normale sans ces chiffres si vous lisez le fichier d'output `.log`.</i></span>

<span style="color:blue"><b>Q2.6.</b><i>Quelle différence observez vous entre le <b>nouveau fichier</b> `.ped` et le fichier `.ped` <b>initial</b> ?</i></span>

*votre réponse*

<div class="alert alert-block alert-warning"><b>En résumé</b><br>
Vous avez appris à:<br>
    - identifier les commandes de base dans PLINK pour lire les fichiers d'inputs<br>
    - ajouter un suffixe aux fichiers d'outputs<br>
    - convertir les fichiers entre les formats plats et binaires.<br>
    
Le rappel de ces commandes, tiré de la liste des commandes PLINK disponibles à <a href="http://zzz.bwh.harvard.edu/plink/reference.shtml">ce lien</a>, est résumé ci-dessous:<br>
    

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-0pky{border-color:inherit;text-align:left;vertical-align:top}
</style>
<table class="tg">
<thead>
  <tr>
    <th class="tg-0pky">Option</th>
    <th class="tg-0pky">Parameter/default</th>
    <th class="tg-0pky">Description</th>
  </tr>
</thead>
<tbody>
  <tr>
    <td class="tg-0pky">Basic input/output</td>
    <td class="tg-0pky"></td>
    <td class="tg-0pky"></td>
  </tr>
  <tr>
    <td class="tg-0pky">--file</td>
    <td class="tg-0pky"> {plink}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Specify .ped and .map files</td>
  </tr>
  <tr>
    <td class="tg-0pky">--ped</td>
    <td class="tg-0pky">&nbsp;&nbsp;{plink.ped}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Specify .ped file</td>
  </tr>
  <tr>
    <td class="tg-0pky">--map</td>
    <td class="tg-0pky">&nbsp;&nbsp;{plink.map}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Specify .map file</td>
  </tr>
  <tr>
    <td class="tg-0pky">--bfile</td>
    <td class="tg-0pky"> {plink}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Specify .bed, .bim and .fam</td>
  </tr>
  <tr>
    <td class="tg-0pky">--bed</td>
    <td class="tg-0pky"> {plink.bed}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Specify .bed file</td>
  </tr>
  <tr>
    <td class="tg-0pky">--bim</td>
    <td class="tg-0pky"> {plink.bim}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Specify .bim file</td>
  </tr>
  <tr>
    <td class="tg-0pky">--fam</td>
    <td class="tg-0pky"> {plink.fam}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Specify .fam file</td>
  </tr>
  <tr>
    <td class="tg-0pky">--out</td>
    <td class="tg-0pky"> {plink}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Specify output root filename</td>
  </tr>
  <tr>
    <td class="tg-0pky">Other data management options</td>
    <td class="tg-0pky"></td>
    <td class="tg-0pky"></td>
  </tr>
  <tr>
    <td class="tg-0pky">--make-bed</td>
    <td class="tg-0pky">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Make .bed, .fam and .bim</td>
  </tr>
  <tr>
    <td class="tg-0pky">--recode</td>
    <td class="tg-0pky">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <td class="tg-0pky">&nbsp;&nbsp;Output new .ped and .map files</td>
  </tr>
</tbody>
</table>
</div>

<div class="alert alert-block alert-success"><b>=> Bravo !</b><br>Vous avez fait un bel effort pour apprendre à utiliser PLINK avec les commandes de base. Vous êtes désormais prêts à travailler avec de vrais fichiers de patients (les cas) et de contrôles. Nettoyez juste avant le dossier « output » de ces fichiers d'entrainement (rm = remove) avec la commande suivante:</div>

In [ ]:
%%bash
# cell 20
rm output/*

Suite du TP dans le prochain notebook!

---
---

<div class="alert alert-block alert-info"> 
    
<b>Rappel</b>, dans un notebook :

- la combinaison de touches <kbd>Ctrl</kbd>+<kbd>Entrée</kbd> exécute une cellule.<br>
- la combinaison de touches <kbd>Shift</kbd>+<kbd>Entrée</kbd> exécute une cellule puis passe à la suivante. C'est équivalent à cliquer sur l'icone ▶️ dans la barre de menu du notebook.<br>
- la combinaison de touches <kbd>Alt</kbd>+<kbd>Entrée</kbd> exécute une cellule puis en crée une nouvelle (vide) en dessous.<br>

Pour ajouter une cellule, vous pouvez aussi cliquer sur l'icone ➕ dans la barre de menu du notebook.
- vous pouvez déplacer les cellules en les glissant pour les réorganiser les unes en-dessous des autres.<br>
- vous pouvez ajouter des commentaires, soit en commençant la ligne par un "#" dans une cellule de code (ces lignes ne seront pas éxécutées), soit dans une nouvelle céllule de type Markdown.<br>
- vous sélectionner le type de cellule dans le menu en haut de votre Vnotebbok.<br>
    - "Code" pour entrer des lignes de commande à éxécuter <br>
    - "Markdown pour ajouter du texte balisé qui peut être formatté<br>
- pour modifier une cellule ̀Markdown, double-cliquez dessus<br>
- attention à la casse des caractères pour les cellules de code<br>
- évitez les caractères spéciaux dans les cellules de code<br>
    
<em>  
To make nice html reports with markdown: <a href="https://dillinger.io/" title="dillinger.io">html visualization tool 1</a> or <a href="https://stackedit.io/app#" title="stackedit.io">html visualization tool 2</a>, <a href="https://www.tablesgenerator.com/markdown_tables" title="tablesgenerator.com">to draw nice tables</a>, and the <a href="https://medium.com/analytics-vidhya/the-ultimate-markdown-guide-for-jupyter-notebook-d5e5abf728fd" title="Ultimate guide">Ultimate guide</a>. <br>
Further reading on JupyterLab notebooks: <a href="https://jupyterlab.readthedocs.io/en/latest/user/notebook.html" title="Jupyter Lab">Jupyter Lab documentation</a>.<br>
Here we are using JupyterLab interface implemented as part of the <a href="https://plasmabio.org/" title="plasmabio.org">Plasmabio</a> project led by Sandrine Caburet, Pierre Poulain and Claire Vandiedonck.
</em>    
</div>

Les informations de configuration de ce notebook et de l'environnement jupyterlab associé sont disponibles à ce lien: https://github.com/CVandiedonck/T1D_GWAS_long_version

*[last version: 18/02/2026 by Claire Vandiedonck]*

---